# Imports

In [1]:
import re

import numpy as np
import pandas as pd
import skfuzzy as fuzz

from sklearn import set_config

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import QuantileTransformer, StandardScaler

from feature_engine.creation import DecisionTreeFeatures

## Utils

In [2]:
set_config(transform_output="pandas")

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 100)

In [3]:
class ClusterFeatureGenerator(BaseEstimator, TransformerMixin):
    
    def __init__(self, n_clusters=5, cols=None, random_state=42, m=2.0):
        self.n_clusters = n_clusters
        self.cols = cols
        self.random_state = random_state
        self.m = m
        
        self.scaler = StandardScaler()
    
    def fit(self, X, y=None):
        X = X.copy()
        
        if self.cols is None:
            self.cols_ = X.select_dtypes(include=np.number).columns
        else:
            self.cols_ = self.cols
        
        X_num = X[self.cols_].fillna(0)
        X_scaled = self.scaler.fit_transform(X_num)
        
        self.cntr_, self.u_, _, _, _, _, _ = fuzz.cluster.cmeans(
            X_scaled.T,
            c=self.n_clusters,
            m=self.m,
            error=1e-5,
            maxiter=1000,
            seed=self.random_state
        )
        
        return self
    
    def transform(self, X):
        X = X.copy()
        
        X_num = X[self.cols_].fillna(0)
        X_scaled = self.scaler.transform(X_num)
        
        u, _, _, _, _, _ = fuzz.cluster.cmeans_predict(
            X_scaled.T,
            self.cntr_,
            m=self.m,
            error=1e-5,
            maxiter=1000
        )
        
        for i in range(self.n_clusters):
            X[f'fuzzy_{i}_{"_".join(self.cols_)}'] = u[i]
        
        return X

In [4]:
class AdaptiveFuzzyTransformer(BaseEstimator, TransformerMixin):
    def __init__(
        self,
        columns=None,
        n_quantiles=1000,
        output_distribution="uniform",
        append=True,
        eps=1e-4,
        handle_degenerate="constant"  # "constant" ou "skip"
    ):
        self.columns = columns
        self.n_quantiles = n_quantiles
        self.output_distribution = output_distribution
        self.append = append
        self.eps = eps
        self.handle_degenerate = handle_degenerate

        self.transformers_ = {}
        self.bounds_ = {}
        self.is_degenerate_ = {}

    def fit(self, X, y=None):
        X = pd.DataFrame(X)

        for col in self.columns:
            qt = QuantileTransformer(
                n_quantiles=min(self.n_quantiles, X.shape[0]),
                output_distribution=self.output_distribution
            )

            q = qt.fit_transform(X[[col]]).values.flatten()

            if np.std(q) < 1e-6:
                self.is_degenerate_[col] = True
                self.transformers_[col] = qt
                self.bounds_[col] = (0.25, 0.5, 0.75)  # dummy
                continue
            else:
                self.is_degenerate_[col] = False

            q25, q50, q75 = np.quantile(q, [0.25, 0.5, 0.75])

            if q25 >= q50:
                q25 = max(0, q50 - self.eps)

            if q75 <= q50:
                q75 = min(1, q50 + self.eps)

            if q25 >= q75:
                q25 = max(0, q50 - self.eps)
                q75 = min(1, q50 + self.eps)

            q25 = np.clip(q25, 0, 1)
            q50 = np.clip(q50, 0, 1)
            q75 = np.clip(q75, 0, 1)

            self.transformers_[col] = qt
            self.bounds_[col] = (q25, q50, q75)

        return self

    def transform(self, X):
        X = pd.DataFrame(X).copy()
        fuzzy_features = []

        for col in self.columns:
            qt = self.transformers_[col]
            q = qt.transform(X[[col]]).values.flatten()

            if self.is_degenerate_[col]:
                if self.handle_degenerate == "skip":
                    continue

                low = np.zeros_like(q)
                medium = np.ones_like(q)
                high = np.zeros_like(q)

            else:
                q25, q50, q75 = self.bounds_[col]

                low = fuzz.trimf(q, [0, 0, q50])
                medium = fuzz.trimf(q, [q25, q50, q75])
                high = fuzz.trimf(q, [q50, 1, 1])

                low = np.nan_to_num(low)
                medium = np.nan_to_num(medium)
                high = np.nan_to_num(high)

            df_fuzzy = pd.DataFrame({
                f"{col}_low": low,
                f"{col}_medium": medium,
                f"{col}_high": high
            }, index=X.index)

            fuzzy_features.append(df_fuzzy)

        if not fuzzy_features:
            return X if self.append else pd.DataFrame(index=X.index)

        fuzzy_df = pd.concat(fuzzy_features, axis=1)

        if self.append:
            return pd.concat([X, fuzzy_df], axis=1)
        else:
            return fuzzy_df

In [5]:
class FeatureEngineeringTransformer(BaseEstimator, TransformerMixin):
    
    def __init__(self, lags=[1, 2, 3, 4, 5, 6], rolling_window=3):
        self.lags = lags
        self.rolling_window = rolling_window
    
    def fit(self, X, y=None):
        return self
    
    def _rolling_slope(self, values):
        """
        Calcula a inclinação (slope) da regressão linear
        nas últimas janelas.
        """
        y = np.array(values)
        x = np.arange(len(y))
        
        if len(y) < 2 or np.isnan(y).all():
            return np.nan
        
        mask = ~np.isnan(y)
        x = x[mask]
        y = y[mask]
        
        if len(y) < 2:
            return np.nan
        
        slope = np.polyfit(x, y, 1)[0]
        return slope
    
    def transform(self, X):
        X = X.copy()
        
        X['lap_time_inv'] = 1 / (X['LapTime (s)'] + 1e-6)
        
        X['degradation_per_lap'] = X['Cumulative_Degradation'] / (X['LapNumber'] + 1e-6)
        
        X['position_norm'] = X.groupby('Race')['Position'].transform(lambda x: x / x.max())
        
        X['position_change_cum'] = X.groupby(['Race', 'Driver'])['Position_Change'].cumsum()
        
        X['is_pit_lap'] = (X['PitStop'] == 1).astype(int)
        
        X['laps_since_pit'] = X.groupby(['Race', 'Driver'])['is_pit_lap'].cumsum()
        
        X['tyre_life_ratio'] = X['TyreLife'] / (X['LapNumber'] + 1e-6)
        
        X['stint_progress'] = X.groupby(['Race', 'Driver', 'Stint']).cumcount()
        
        X['compound_tyre_life'] = X['TyreLife'] * X['Compound'].astype('category').cat.codes
        
        X['delta_x_tyre_life'] = X['LapTime_Delta'] * X['TyreLife']
        
        X['driver_mean_lap'] = X.groupby('Driver')['LapTime (s)'].transform('mean')
        
        X['race_progress_sin'] = np.sin(X['RaceProgress'] * np.pi)
        
        X = X.sort_values(['Race', 'Driver', 'LapNumber'])
        
        # for lag in self.lags:
        #     X[f'lap_time_lag_{lag}'] = X.groupby(['Race', 'Driver'])['LapTime (s)'].shift(lag)
        #     X[f'position_lag_{lag}'] = X.groupby(['Race', 'Driver'])['Position'].shift(lag)
        
        # Diferença entre lags
        # X["delta_lag1_lag2"] = X["lap_time_lag_1"] - X["lap_time_lag_2"]
        
        # Segunda derivada temporal
        # X["lap_acceleration"] = (
        #     X["lap_time_lag_1"]
        #     - 2 * X["lap_time_lag_2"]
        #     + X["lap_time_lag_3"]
        # )
        
        # X['lap_time_trend_3'] = X['LapTime (s)'] - X.groupby(['Race', 'Driver'])['LapTime (s)'].shift(3)
        
        # X[f'lap_time_roll_mean_{self.rolling_window}'] = (
        #     X.groupby(['Race', 'Driver'])['LapTime (s)']
        #     .rolling(self.rolling_window)
        #     .mean()
        #     .reset_index(level=[0,1], drop=True)
        # )
        
        # X[f'lap_time_roll_std_{self.rolling_window}'] = (
        #     X.groupby(['Race', 'Driver'])['LapTime (s)']
        #     .rolling(self.rolling_window)
        #     .std()
        #     .reset_index(level=[0,1], drop=True)
        # )
        
        # X['lap_time_slope_5'] = (
        #     X.groupby(['Race', 'Driver'])['LapTime (s)']
        #     .rolling(5)
        #     .apply(self._rolling_slope, raw=False)
        #     .reset_index(level=[0,1], drop=True)
        # )
        
        X['lap_time_vs_race_mean'] = X['LapTime (s)'] - X.groupby(['Race', 'LapNumber'])['LapTime (s)'].transform('mean')
        
        X['position_vs_mean'] = X['Position'] - X.groupby(['Race', 'LapNumber'])['Position'].transform('mean')
        
        X['lap_rank_pct'] = X.groupby(['Race', 'LapNumber'])['LapTime (s)'].rank(pct=True)
        
        compound_mean = X.groupby(['Race', 'LapNumber', 'Compound'])['LapTime (s)'].transform('mean')
        
        X['pace_vs_compound'] = X['LapTime (s)'] - compound_mean
        
        X = X.sort_values(['Race', 'LapNumber', 'Position'])
        
        # X['position_ahead'] = X.groupby(['Race', 'LapNumber'])['Position'].shift(1)
        
        # X['position_behind'] = X.groupby(['Race', 'LapNumber'])['Position'].shift(-1)
        
        # X['gap_ahead'] = X['Position'] - X['position_ahead']
        
        # X['gap_behind'] = X['position_behind'] - X['Position']
        
        # X['is_in_traffic'] = ((X['gap_ahead'].abs() <= 1) | (X['gap_behind'].abs() <= 1)).astype(int)
        
        # X['degradation_acceleration'] = X.groupby(['Race', 'Driver'])['Cumulative_Degradation'].diff()
        
        X['lap_time_x_tyre'] = X['LapTime (s)'] * X['TyreLife']
        
        X['position_x_progress'] = X['Position'] * X['RaceProgress']
        
        X['degradation_x_progress'] = X['Cumulative_Degradation'] * X['RaceProgress']
        
        X['race_progress_squared'] = X['RaceProgress'] ** 2
        
        X['driver_avg_position'] = X.groupby('Driver')['Position'].transform('mean')
        
        X = X.replace([np.inf, -np.inf], np.nan)
        
        X = X.sort_values(by='id')
        
        return X

In [6]:
def clean_columns(df):
    df.columns = [
        re.sub(r'[^A-Za-z0-9_]', '', col.replace(' ', '_')).lower()
        for col in df.columns
    ]
    return df

# Loading Dataset

In [9]:
df_train = pd.read_csv('../data/raw/train.csv', index_col='id')
df_test = pd.read_csv('../data/raw/test.csv', index_col='id')

In [10]:
df_train.head()

,Driver,Compound,Race,Year,PitStop,LapNumber,Stint,TyreLife,Position,LapTime (s),LapTime_Delta,Cumulative_Degradation,RaceProgress,Position_Change,PitNextLap
id,,,,,,,,,,,,,,,
0,D109,HARD,Canadian Grand Prix,2022,0,50,2,39.0,8,78.491,-7.564,21.019,0.714286,5.0,1.0
1,D086,HARD,Dutch Grand Prix,2025,1,27,2,7.0,4,75.095,-32.617,-223.207,0.346154,-3.0,0.0
2,ZON,HARD,Austrian Grand Prix,2022,0,59,3,22.0,13,70.945,-7.540,-100.529,0.819444,3.0,1.0
3,SPE,MEDIUM,Pre-Season Testing,2023,0,2,1,2.0,7,94.361,-7.324,-7.324,0.076923,0.0,0.0
4,D019,HARD,Azerbaijan Grand Prix,2022,1,26,3,6.0,2,107.878,8.965,-14.139,0.361111,3.0,0.0


In [11]:
df_train.tail()

,Driver,Compound,Race,Year,PitStop,LapNumber,Stint,TyreLife,Position,LapTime (s),LapTime_Delta,Cumulative_Degradation,RaceProgress,Position_Change,PitNextLap
id,,,,,,,,,,,,,,,
439135,D755,MEDIUM,Miami Grand Prix,2023,0,49,2,8.0,17,92.638,-0.076,-15.859,0.859649,0.0,0.0
439136,D731,MEDIUM,Miami Grand Prix,2023,0,49,2,5.0,1,85.890,-0.083,-4.907,0.859649,0.0,0.0
439137,D716,MEDIUM,Miami Grand Prix,2023,0,49,2,18.0,1,91.644,-0.182,-56.371,0.942308,0.0,0.0
439138,D665,HARD,Abu Dhabi Grand Prix,2023,0,48,3,10.0,1,89.947,-0.001,-20.721,0.827586,1.0,0.0
439139,D714,HARD,Miami Grand Prix,2023,0,49,2,32.0,1,91.691,-0.123,-15.828,0.859649,0.0,0.0


## Columns

In [12]:
df_train.columns

Index(['Driver', 'Compound', 'Race', 'Year', 'PitStop', 'LapNumber', 'Stint',
       'TyreLife', 'Position', 'LapTime (s)', 'LapTime_Delta',
       'Cumulative_Degradation', 'RaceProgress', 'Position_Change',
       'PitNextLap'],
      dtype='str')

In [13]:
df_test.columns

Index(['Driver', 'Compound', 'Race', 'Year', 'PitStop', 'LapNumber', 'Stint',
       'TyreLife', 'Position', 'LapTime (s)', 'LapTime_Delta',
       'Cumulative_Degradation', 'RaceProgress', 'Position_Change'],
      dtype='str')

## Data Dimension

In [14]:
df_train.shape

(439140, 15)

In [15]:
df_test.shape

(188165, 14)

## Check NA

In [16]:
df_train.isna().mean()

Driver                    0.0
Compound                  0.0
Race                      0.0
Year                      0.0
PitStop                   0.0
LapNumber                 0.0
Stint                     0.0
TyreLife                  0.0
Position                  0.0
LapTime (s)               0.0
LapTime_Delta             0.0
Cumulative_Degradation    0.0
RaceProgress              0.0
Position_Change           0.0
PitNextLap                0.0
dtype: float64

In [17]:
df_test.isna().mean()

Driver                    0.0
Compound                  0.0
Race                      0.0
Year                      0.0
PitStop                   0.0
LapNumber                 0.0
Stint                     0.0
TyreLife                  0.0
Position                  0.0
LapTime (s)               0.0
LapTime_Delta             0.0
Cumulative_Degradation    0.0
RaceProgress              0.0
Position_Change           0.0
dtype: float64

## Splitting X and y

In [18]:
X_train = df_train.drop(['PitNextLap'], axis=1)
y_train = df_train.loc[:, ['PitNextLap']].astype(int)

X_test = df_test.copy()

# Feature Engineering

## Feature Engineering Transform

In [19]:
fet = FeatureEngineeringTransformer()

X_train = fet.fit_transform(X_train)
X_test = fet.transform(X_test)

## Clean Column Names

In [20]:
X_train = clean_columns(X_train)
X_test = clean_columns(X_test)

## Fuzzy Logic

In [21]:
fuzzy_transform = AdaptiveFuzzyTransformer(columns=['lapnumber', 'stint', 'position', 'laptime_s', 'raceprogress'])

X_train = fuzzy_transform.fit_transform(X_train)
X_test = fuzzy_transform.transform(X_test)

## Clustering Feature Engineering

In [22]:
clustering_lap_time = ClusterFeatureGenerator(n_clusters=3, cols=['laptime_s', 'laptime_delta'])

X_train = clustering_lap_time.fit_transform(X_train)
X_test = clustering_lap_time.transform(X_test)

In [23]:
clustering_tyre_degradation = ClusterFeatureGenerator(n_clusters=3, cols=['tyrelife', 'cumulative_degradation'])

X_train = clustering_tyre_degradation.fit_transform(X_train)
X_test = clustering_tyre_degradation.transform(X_test)

In [24]:
clustering_position = ClusterFeatureGenerator(n_clusters=3, cols=['position', 'position_change'])

X_train = clustering_position.fit_transform(X_train)
X_test = clustering_position.transform(X_test)

In [25]:
clustering_race_lap = ClusterFeatureGenerator(n_clusters=3, cols=['raceprogress', 'lapnumber'])

X_train = clustering_race_lap.fit_transform(X_train)
X_test = clustering_race_lap.transform(X_test)

## Decision Tree Features

In [26]:
dtf = DecisionTreeFeatures(
    variables=['raceprogress', 'position', 'laptime_s', 'cumulative_degradation'], 
    regression=False, 
    features_to_combine=2,
    missing_values='ignore'
)

X_train = dtf.fit_transform(X_train, y_train.PitNextLap)
X_test = dtf.transform(X_test)

## Change Type

In [27]:
cols = ['driver', 'compound', 'race']

for col in cols:
    X_train[col] = X_train[col].astype('category')
    X_test[col] = X_test[col].astype('category')


X_train = clean_columns(X_train)
X_test = clean_columns(X_test)

# Exporting Datasets

In [29]:
X_train.to_parquet('../data/interim/X_train.parquet')
X_test.to_parquet('../data/interim/X_test.parquet')

y_train.to_parquet('../data/processed/y_train.parquet')